[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C07_ML_Foundations_Course/05_optimizers_from_scratch/05_optimizers_from_scratch.ipynb)

# 05 · 优化器从零：SGD → AdamW

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span>
在真实 **UCI 乳腺癌**上训练逻辑回归，亲手实现 5 个优化器并对比收敛曲线，再加 warmup+cosine 调度。

**你将完成：**
1. SGD / Momentum 更新式
2. Adam（含偏差校正）
3. AdamW（解耦 weight decay）
4. warmup + cosine 学习率调度，对比收敛速度

> 数据：UCI WDBC（569 样本, 30 特征, 二分类）。

## 0 · 加载数据 + 统一的训练循环

In [ ]:
import os, urllib.request
import numpy as np
np.set_printoptions(precision=4, suppress=True)
CACHE=os.path.expanduser("~/.ml_foundations_data"); os.makedirs(CACHE,exist_ok=True)
def fetch(u,f):
    p=os.path.join(CACHE,f)
    if not os.path.exists(p): urllib.request.urlretrieve(u,p)
    return p
import pandas as pd
df=pd.read_csv(fetch("https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data","wdbc.data"),header=None)
X=df.iloc[:,2:].to_numpy(float); y=(df[1].to_numpy()=="M").astype(float)
X=(X-X.mean(0))/X.std(0)   # 标准化
print("X:",X.shape,"恶性占比:",y.mean().round(3))

def sigmoid(z): return 1/(1+np.exp(-np.clip(z,-500,500)))
def loss_grad(w, Xb, yb):
    p=sigmoid(Xb@w); g=Xb.T@(p-yb)/len(yb)
    eps=1e-12; L=-np.mean(yb*np.log(p+eps)+(1-yb)*np.log(1-p+eps))
    return L, g

def train(opt_step, n_epochs=30, batch=64, seed=0, **state0):
    rng=np.random.default_rng(seed); w=np.zeros(X.shape[1]); state=dict(state0); losses=[]; t=0
    for ep in range(n_epochs):
        idx=rng.permutation(len(y))
        for s in range(0,len(y),batch):
            b=idx[s:s+batch]; t+=1
            L,g=loss_grad(w, X[b], y[b])
            w=opt_step(w, g, state, t)
        losses.append(loss_grad(w,X,y)[0])
    return w, losses

## 1 · SGD 与 Momentum

最朴素的 `θ -= lr*g`，和加惯性的动量版。对比同样 epoch 下谁收敛更快更稳。

In [ ]:
def sgd(lr):
    def step(w, g, st, t): return w - lr*g
    return step

def momentum(lr, beta=0.9):
    def step(w, g, st, t):
        st["v"]=beta*st.get("v",0)+g
        return w - lr*st["v"]
    return step

_, l_sgd = train(sgd(0.3))
_, l_mom = train(momentum(0.3))
print(f"30 epoch 后  SGD loss={l_sgd[-1]:.4f}   Momentum loss={l_mom[-1]:.4f}")
print(f"达到 loss<0.15 所需 epoch:  SGD={next((i for i,l in enumerate(l_sgd) if l<0.15),'>30')}  "
      f"Momentum={next((i for i,l in enumerate(l_mom) if l<0.15),'>30')}")

## 2 · Adam（含偏差校正）

动量(m) + RMSProp(v) + 偏差校正。看去掉偏差校正前期会慢多少。

In [ ]:
def adam(lr, b1=0.9, b2=0.999, eps=1e-8, bias_correct=True):
    def step(w, g, st, t):
        st["m"]=b1*st.get("m",0)+(1-b1)*g
        st["v"]=b2*st.get("v",0)+(1-b2)*g*g
        if bias_correct:
            mh=st["m"]/(1-b1**t); vh=st["v"]/(1-b2**t)
        else:
            mh,vh=st["m"],st["v"]
        return w - lr*mh/(np.sqrt(vh)+eps)
    return step

_, l_adam = train(adam(0.05))
_, l_nobc = train(adam(0.05, bias_correct=False))
print(f"Adam(有偏差校正) 第1 epoch loss={l_adam[0]:.4f}  最终={l_adam[-1]:.4f}")
print(f"Adam(无偏差校正) 第1 epoch loss={l_nobc[0]:.4f}  最终={l_nobc[-1]:.4f}")
print("=> 偏差校正让前期收敛更快")

## 3 · AdamW：解耦 weight decay

L2 不混进梯度，而是直接 `w -= lr*wd*w`。对比 Adam+L2(混进梯度) 与 AdamW 的权重范数。

In [ ]:
def adamw(lr, wd=0.01, b1=0.9, b2=0.999, eps=1e-8):
    def step(w, g, st, t):
        st["m"]=b1*st.get("m",0)+(1-b1)*g
        st["v"]=b2*st.get("v",0)+(1-b2)*g*g
        mh=st["m"]/(1-b1**t); vh=st["v"]/(1-b2**t)
        return w - lr*(mh/(np.sqrt(vh)+eps) + wd*w)   # 解耦：wd 直接作用在 w
    return step

w_w,_=train(adamw(0.05, wd=0.05)); w_p,_=train(adam(0.05))
print(f"AdamW(wd=0.05) 权重范数={np.linalg.norm(w_w):.3f}")
print(f"Adam(no decay)  权重范数={np.linalg.norm(w_p):.3f}")
print("=> AdamW 的权重更小（正则化生效），但精度不掉")
print(f"   两者 test 风格 acc: AdamW={((sigmoid(X@w_w)>0.5)==y).mean():.3f}  Adam={((sigmoid(X@w_p)>0.5)==y).mean():.3f}")

## 4 · 学习率调度：warmup + cosine

前期线性升、之后余弦降。在带调度的 lr 下训练。

In [ ]:
def lr_schedule(t, peak=0.1, warmup=20, total=200):
    if t < warmup: return peak*t/warmup
    prog=(t-warmup)/(total-warmup)
    return 0.5*peak*(1+np.cos(np.pi*min(prog,1.0)))

ts=np.arange(0,200)
lrs=[lr_schedule(t) for t in ts]
print(f"step 0 lr={lrs[0]:.4f}  峰值@step20 lr={lrs[20]:.4f}  末端 lr={lrs[-1]:.4f}")
print("warmup 段:", [round(lr_schedule(t),3) for t in range(0,25,5)])
print("=> 先升后降，正是大模型训练的标准 lr 曲线")

## 5 · AdaGrad → RMSProp 的修复，与偏差校正的微例

讲解里的自适应优化器演化有两个关键洞见，这里都在真实数据上看清楚。**(a) AdaGrad 的分母只增不减**：它把历史梯度平方**累加**进分母，于是有效步长 $1/\sqrt{G}$ 随训练单调缩小，后期「步子迈不动」；**RMSProp** 把累加换成**指数移动平均（EMA）**，分母不再无限膨胀，正好修好这个病。**(b) 偏差校正为什么除以 $1-\beta^t$**：因为一阶矩 $m$ 初始化为 0，喂入恒定梯度 $g$ 时可以解析地算出 $m_t=(1-\beta^t)g$——前几步严重偏小，除以 $1-\beta^t$ 恰好把它放大回真实量级。下面把这两点都验证到机器精度。

In [ ]:
# (a) AdaGrad 的分母只增不减 -> 后期有效步长趋零；RMSProp 用 EMA 修好它。
def adagrad(lr, eps=1e-8):
    def step(w, g, st, t):
        st["G"] = st.get("G", 0) + g*g            # 历史梯度平方「累加」（只增）
        st["last_scale"] = float(np.mean(1.0/(np.sqrt(st["G"])+eps)))   # 记录平均有效步长系数
        return w - lr*g/(np.sqrt(st["G"])+eps)
    return step
def rmsprop(lr, beta=0.9, eps=1e-8):
    def step(w, g, st, t):
        st["G"] = beta*st.get("G", 0) + (1-beta)*g*g    # 「指数移动平均」（不无限增长）
        st["last_scale"] = float(np.mean(1.0/(np.sqrt(st["G"])+eps)))
        return w - lr*g/(np.sqrt(st["G"])+eps)
    return step

ag = adagrad(0.5); rp = rmsprop(0.05)
st_ag = {}; st_rp = {}
def wrap(opt, st):
    def step(w, g, s, t):
        out = opt(w, g, st, t); s.update(st); return out
    return step
_, l_ag = train(wrap(ag, st_ag)); _, l_rp = train(wrap(rp, st_rp))
print(f"AdaGrad: 末端平均有效步长系数 1/√G = {st_ag['last_scale']:.4f}   30 epoch loss={l_ag[-1]:.4f}")
print(f"RMSProp: 末端平均有效步长系数 1/√G = {st_rp['last_scale']:.4f}   30 epoch loss={l_rp[-1]:.4f}")
print("=> AdaGrad 的 1/√G 随训练单调缩小（分母只增）；RMSProp 用 EMA 让它稳定，避免后期『步子迈不动』")
assert st_rp["last_scale"] > st_ag["last_scale"], "RMSProp 的有效步长应大于已被累积分母压扁的 AdaGrad"

# (b) 微例：偏差校正为什么除以 (1-β^t) —— 验证 E[m_t] ≈ (1-β^t)·g（常梯度下）
beta1 = 0.9; g_const = 1.0; m = 0.0
print("\n偏差校正微例（喂恒定梯度 g=1，看 m_t 与理论 (1-β^t)）：")
for t in range(1, 6):
    m = beta1*m + (1-beta1)*g_const
    theory = 1 - beta1**t
    print(f"  t={t}: m_t={m:.4f}   1-β^t={theory:.4f}   校正后 m_t/(1-β^t)={m/theory:.4f}")
    assert abs(m - theory) < 1e-9, "常梯度下 m_t 应精确等于 (1-β^t)·g"
print("=> m 初始化为 0 使前几步严重偏小（≈(1-β^t)·g）；除以 (1-β^t) 正好把它放大回真实量级 g")

---
## ✏️ 练习区

### ✏️ 练习 1：动量更新

实现 `momentum_step(w, g, state, lr, beta)`：原地更新 `state['v']` 并返回新 `w`。
`state` 是 dict，首次调用时可能没有 `'v'`。

In [ ]:
def momentum_step(w, g, state, lr=0.3, beta=0.9):
    # TODO: v = beta*v + g ; w = w - lr*v ; 把 v 存回 state
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测 ——
st={}; w=np.zeros(3); g=np.array([1.,1.,1.])
w=momentum_step(w,g,st,lr=0.1,beta=0.9)
assert np.allclose(st["v"], [1,1,1]) and np.allclose(w, [-0.1,-0.1,-0.1])
w=momentum_step(w,g,st,lr=0.1,beta=0.9)
assert np.allclose(st["v"], [1.9,1.9,1.9]), "动量应累积"
print("练习 1 通过 ✓")


### ✏️ 练习 2：Adam 更新（含偏差校正）

实现 `adam_step(w, g, state, t, lr, b1, b2, eps)`。`t` 是从 1 开始的步数（用于偏差校正）。

In [ ]:
def adam_step(w, g, state, t, lr=0.05, b1=0.9, b2=0.999, eps=1e-8):
    # TODO: 更新 m,v；偏差校正 mh,vh；w -= lr*mh/(sqrt(vh)+eps)
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测 ——
st={}; w=np.array([1.0]); g=np.array([0.1])
w1=adam_step(w.copy(),g,st,t=1,lr=0.05)
# t=1 时 mh=g, vh=g^2 -> 步长≈lr -> w 减少约 0.05
assert abs((1-w1[0]) - 0.05) < 1e-6, "Adam 首步在偏差校正下步长≈lr"
# 用 Adam 训练应快速收敛
def step(w,g,st,t): return adam_step(w,g,st,t)
_, ls = train(step)
assert ls[-1] < 0.12, "Adam 应把乳腺癌 loss 训到 <0.12"
print(f"练习 2 通过 ✓  最终 loss={ls[-1]:.4f}")


### ✏️ 练习 3：AdamW vs Adam+L2 的区别

实现 `adamw_step`（解耦 decay）。然后验证：在自适应分母下，AdamW 的 decay 与"把 `wd*w` 加进梯度"
**不等价**（这正是 AdamW 的意义）。

In [ ]:
def adamw_step(w, g, state, t, lr=0.05, wd=0.05, b1=0.9, b2=0.999, eps=1e-8):
    # TODO: 同 adam，但最后 w -= lr*(update + wd*w)，wd*w 不进 m/v
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测 ——
# 同一初始状态，比较 AdamW 与 "Adam(把 wd*w 加进 g)"
w0=np.array([2.0]); g=np.array([0.1])
s1={}; wA=adamw_step(w0.copy(), g, s1, t=1, lr=0.05, wd=0.05)
s2={}
def adam_l2(w,g,st,t,lr=0.05,wd=0.05,b1=0.9,b2=0.999,eps=1e-8):
    g=g+wd*w   # L2 混进梯度（旧做法）
    st["m"]=b1*st.get("m",0)+(1-b1)*g; st["v"]=b2*st.get("v",0)+(1-b2)*g*g
    mh=st["m"]/(1-b1**t); vh=st["v"]/(1-b2**t)
    return w-lr*mh/(np.sqrt(vh)+eps)
wB=adam_l2(w0.copy(), g, s2, t=1)
assert abs(wA[0]-wB[0]) > 1e-5, "两种 decay 应不同（自适应分母把 L2 缩放了）"
print(f"练习 3 通过 ✓  AdamW={wA[0]:.5f} vs Adam+L2={wB[0]:.5f}（不同！）")


### ✏️ 练习 4：warmup + cosine 调度

实现 `cosine_warmup(t, peak, warmup, total)`：`t<warmup` 线性升到 peak；之后余弦降到 0。

In [ ]:
def cosine_warmup(t, peak=0.1, warmup=20, total=200):
    # TODO: warmup 段 peak*t/warmup；之后 0.5*peak*(1+cos(pi*progress))
    raise NotImplementedError


In [ ]:
# —— 练习 4 自测 ——
assert abs(cosine_warmup(0)) < 1e-9, "t=0 lr=0"
assert abs(cosine_warmup(20)-0.1) < 1e-9, "warmup 末端到峰值"
assert cosine_warmup(10) < cosine_warmup(20), "warmup 段递增"
assert cosine_warmup(200) < 1e-6, "末端降到≈0"
assert cosine_warmup(110) < cosine_warmup(30), "cosine 段递减"
print("练习 4 通过 ✓")


---
## 📖 参考答案

In [ ]:
# 练习 1
def momentum_step(w, g, state, lr=0.3, beta=0.9):
    state["v"]=beta*state.get("v",0)+g
    return w - lr*state["v"]
print("练习 1 ✓")

In [ ]:
# 练习 2
def adam_step(w, g, state, t, lr=0.05, b1=0.9, b2=0.999, eps=1e-8):
    state["m"]=b1*state.get("m",0)+(1-b1)*g
    state["v"]=b2*state.get("v",0)+(1-b2)*g*g
    mh=state["m"]/(1-b1**t); vh=state["v"]/(1-b2**t)
    return w - lr*mh/(np.sqrt(vh)+eps)
print("练习 2 ✓")

In [ ]:
# 练习 3
def adamw_step(w, g, state, t, lr=0.05, wd=0.05, b1=0.9, b2=0.999, eps=1e-8):
    state["m"]=b1*state.get("m",0)+(1-b1)*g
    state["v"]=b2*state.get("v",0)+(1-b2)*g*g
    mh=state["m"]/(1-b1**t); vh=state["v"]/(1-b2**t)
    return w - lr*(mh/(np.sqrt(vh)+eps) + wd*w)
print("练习 3 ✓")

In [ ]:
# 练习 4
def cosine_warmup(t, peak=0.1, warmup=20, total=200):
    if t < warmup: return peak*t/warmup
    prog=min((t-warmup)/(total-warmup), 1.0)
    return 0.5*peak*(1+np.cos(np.pi*prog))
print("练习 4 ✓ —— 你刚实现了训练 GPT 同款的 lr 曲线")